In [2]:
!pip install gradio wikipedia-api sentence-transformers fiftyone -q

**Setup and model load**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import importlib.util
import sys

spec  = importlib.util.spec_from_file_location("utils", "/content/drive/MyDrive/ML_aircraft_recognition/utils.py")
utils = importlib.util.module_from_spec(spec)
sys.modules["utils"] = utils
spec.loader.exec_module(utils)

from utils import load_dataset, get_dataloaders, load_model, load_resnet

import torch
import numpy as np
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = load_dataset()
_, _, _, num_classes, _, _, _ = get_dataloaders(dataset)

model_eff = load_model(num_classes, device)
model_res = load_resnet(num_classes, device)
label_names = sorted(dataset.distinct("ground_truth.label"))

print(f"✅ Models loaded | Device: {device} | Classes: {num_classes}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 100% |███████████████| 3334/3334 [3.6s elapsed, 0s remaining, 1.2K samples/s]      


INFO:eta.core.utils: 100% |███████████████| 3334/3334 [3.6s elapsed, 0s remaining, 1.2K samples/s]      


Loaded 3334 samples
Train: 2333 | Val: 500 | Test: 501 | Classes: 100
Model loaded from /content/drive/MyDrive/ML_aircraft_recognition/best_model.pth
ResNet loaded from /content/drive/MyDrive/ML_aircraft_recognition/best_resnet.pth
✅ Models loaded | Device: cuda | Classes: 100


**Inference function**

In [4]:
inference_transform = transforms.Compose([
    transforms.Resize((255, 255)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def predict(image, models, label_names, device, top_k=5):
    if isinstance(image, str):
        image = Image.open(image).convert("RGB")
    else:
        image = image.convert("RGB")

    tensor = inference_transform(image).unsqueeze(0).to(device)
    probs  = torch.zeros(1, len(label_names)).to(device)

    for model in models:
        model.eval()
        with torch.no_grad():
            probs += torch.softmax(model(tensor), dim=1)
    probs /= len(models)

    top_probs, top_indices = probs.topk(top_k, dim=1)
    top_probs   = top_probs.squeeze().cpu().numpy()
    top_indices = top_indices.squeeze().cpu().numpy()

    return [(label_names[i], float(p)) for i, p in zip(top_indices, top_probs)]

**Wikipedia RAG**

In [5]:
import wikipediaapi
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")
wiki     = wikipediaapi.Wikipedia(language="en", user_agent="aircraft-chatbot/1.0")

def fetch_wikipedia(aircraft_name):
    page = wiki.page(aircraft_name)
    if not page.exists():
        # Try a broader search
        page = wiki.page(f"{aircraft_name} aircraft")
    if not page.exists():
        return None
    return page.text

def chunk_text(text, chunk_size=500, overlap=50):
    words  = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

def build_rag(aircraft_name):
    text = fetch_wikipedia(aircraft_name)
    if not text:
        return None, None
    chunks     = chunk_text(text)
    embeddings = embedder.encode(chunks, convert_to_numpy=True)
    return chunks, embeddings

def retrieve(query, chunks, embeddings, top_k=3):
    query_emb = embedder.encode([query], convert_to_numpy=True)
    scores    = np.dot(embeddings, query_emb.T).squeeze()
    top_idx   = np.argsort(scores)[::-1][:top_k]
    return [chunks[i] for i in top_idx]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Mistral chatbot**

In [1]:
from mistralai import Mistral
print("✅ OK")

✅ OK


In [6]:
from google.colab import userdata

MISTRAL_API_KEY = userdata.get("MISTRAL_KEY")
mistral_client  = Mistral(api_key=MISTRAL_API_KEY)

# Global state
current_aircraft = None
rag_chunks       = None
rag_embeddings   = None
chat_history     = []

def load_aircraft_context(aircraft_name):
    global current_aircraft, rag_chunks, rag_embeddings, chat_history
    current_aircraft = aircraft_name
    chat_history     = []
    rag_chunks, rag_embeddings = build_rag(aircraft_name)
    if rag_chunks:
        return f"✅ Loaded Wikipedia context for **{aircraft_name}**. You can now ask questions!"
    return f"⚠️ No Wikipedia article found for {aircraft_name}. Chatbot will use general knowledge."

def chat(user_message, history):
    global chat_history

    if not current_aircraft:
        return history + [[user_message, "Please upload an image first to identify the aircraft."]]

    # Retrieve relevant chunks
    context = ""
    if rag_chunks and rag_embeddings is not None:
        relevant = retrieve(user_message, rag_chunks, rag_embeddings)
        context  = "\n\n".join(relevant)

    # Build system prompt
    system_prompt = f"""You are an aviation assistant.
The user is asking about the {current_aircraft}.
Use the following Wikipedia context to answer accurately.
If the context does not contain the answer, use your general knowledge.

Context:
{context}"""

    # Build messages
    messages = [{"role": "system", "content": system_prompt}]
    for human, assistant in chat_history:
        messages.append({"role": "user",      "content": human})
        messages.append({"role": "assistant", "content": assistant})
    messages.append({"role": "user", "content": user_message})

    response = mistral_client.chat.complete(
      model    = "mistral-small-latest",
      messages = messages
    )

    reply = response.choices[0].message.content
    chat_history.append((user_message, reply))

    return history + [[user_message, reply]]

**Gradio**

In [8]:
import gradio as gr

def on_image_upload(image):
    if image is None:
        return "No image uploaded.", None

    results = predict(image, [model_eff, model_res], label_names, device)
    top_label, top_conf = results[0]

    fig, ax = plt.subplots(figsize=(8, 4))
    labels  = [r[0] for r in results]
    scores  = [r[1] for r in results]
    colors  = ["green"] + ["steelblue"] * (len(labels) - 1)
    ax.barh(labels[::-1], scores[::-1], color=colors[::-1])
    ax.set_title(f"Top Prediction: {top_label} ({top_conf*100:.1f}%)")
    ax.set_xlabel("Confidence")
    ax.set_xlim(0, 1)
    for i, score in enumerate(scores[::-1]):
        ax.text(score + 0.01, i, f"{score*100:.1f}%", va="center")
    plt.tight_layout()

    status = load_aircraft_context(top_label)
    return status, fig

with gr.Blocks(title="Aircraft Recognition") as app:
    gr.Markdown("# ✈️ Aircraft Recognition & Chatbot")

    with gr.Tab("🔍 Identify Aircraft"):
        with gr.Row():
            image_input  = gr.Image(type="pil", label="Upload Aircraft Photo")
            output_chart = gr.Plot(label="Predictions")
        rag_status = gr.Markdown("")
        image_input.change(
            fn=on_image_upload,
            inputs=image_input,
            outputs=[rag_status, output_chart]
        )

    with gr.Tab("💬 Ask About the Aircraft"):
        gr.Markdown("Upload an image first, then ask anything about the identified aircraft.")
        chatbot   = gr.Chatbot(height=400)
        msg_input = gr.Textbox(placeholder="Ask something about the aircraft...", label="Your question")
        clear_btn = gr.Button("Clear chat")

        msg_input.submit(fn=chat, inputs=[msg_input, chatbot], outputs=chatbot)
        clear_btn.click(fn=lambda: [], outputs=chatbot)

app.launch(share=True)

/tmp/ipykernel_20870/2904376275.py:41: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot   = gr.Chatbot(height=400)
/tmp/ipykernel_20870/2904376275.py:41: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot   = gr.Chatbot(height=400)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a0f021c01ac031ec39.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
